# 10 — SP-LRP Evaluation — Semantic-Prior Guided LRP (ViT-Base/16, CRC Histology)

Tests a CRC-adapted **SP-LRP** (Semantic-Prior Guided Layer-Wise Relevance Propagation)
against Chefer Transformer-Attribution on a trained **ViT-Base/16**, CRC-VAL-HE-7K test
set (9 classes), using **Insertion / Deletion AUC**.

## What SP-LRP does

SP-LRP = Chefer's Transformer-Attribution relevance, but the raw attention `A^(l)` is
replaced by a **morphology-weighted** attention
`Â^(l) = rownorm(A^(l) ⊙ M^(l))`, with `M^(l)_{ij} = exp(−‖sᵢ−sⱼ‖²/τ_l)`,
`sᵢ` a per-patch semantic-prior vector, and `τ_l = τ₀·(l+1)` growing with depth.
When `M = 1` this reduces **exactly** to Chefer Transformer-Attribution — the paper's
`Standard ViT-LRP` (w/o MSPM) baseline.

## Adaptation notes (read before running)

1. **Prior source = ResNet-50, not DenseNet-201.** CLAUDE.md forbids DenseNet; we
   substitute your authorised CRC-trained **ResNet-50** applied densely
   (`DenseMorphologyPrior`) to produce per-patch 9-class probability vectors. Still an
   independent CNN morphology extractor, per the paper's intent.
2. **The mechanism may be near-degenerate on CRC.** SP-LRP suppresses relevance flow
   *between different tissue types within one image*. That assumes a field of view
   spanning multiple tissues (the paper's gigapixel WSIs). NCT-CRC-HE tiles are
   pre-cropped **single-class** 224px patches, so intra-tile heterogeneity is limited
   and `M ≈ 1` (SP-LRP ≈ Chefer TA). Expect a small effect; the test quantifies it.
3. **Insertion/Deletion only.** Pointing Game, MAS, DSC need lesion masks CRC lacks.

## Ablation grid (mirrors the paper's Table 2)

| Variant | Modulation |
|---|---|
| `chefer_ta` | off (== Standard ViT-LRP baseline) |
| `splrp_full` | real prior, layer-dependent τ |
| `splrp_fixed_tau` | real prior, fixed τ₀ |
| `splrp_random_prior` | random prior (null control — should ≈ chefer_ta) |

**Edit only Cell 2** (`USER CONFIG`). **Environment**: Kaggle GPU T4 / P100 — Phase 1 only.

⚠ Compute-heavy: SP-LRP needs a backward pass per image, plus insertion/deletion sweeps.
Start with a small `N_IMAGES_PER_CLASS`.

In [ ]:
# 0. Kaggle setup
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru
!pip install -q captum
!pip install -q PyDrive2
!pip install -q scikit-learn

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

In [ ]:
# ============================================================
# USER CONFIG — edit this cell only
# ============================================================

MODEL_NAME = "vit_base"   # this notebook targets ViT-Base/16 CLS-token readout

DRIVE_FOLDER_ID = "190QMsyYCtNo5xXOEm-CWWQCDm4jfPGsu"  # vit_base results folder <-- update as needed

# Dataset subset (SP-LRP is compute-heavy: 1 backward/image + insertion/deletion)
N_IMAGES_PER_CLASS = 15
BATCH_SIZE          = 8
NUM_WORKERS         = 2
IMAGE_SIZE          = 224
SEED                = 42

# SP-LRP hyperparameters
TAU0            = 0.1        # base temperature (paper: 0.1)
LAYER_DEP_TAU   = True       # tau_l = tau0*(l+1) with depth
START_LAYER     = 0

VARIANTS = [
    "chefer_ta",            # baseline: modulation off (== Standard ViT-LRP)
    "splrp_full",           # real prior + layer-dependent tau
    "splrp_fixed_tau",      # real prior + fixed tau0
    "splrp_random_prior",   # random prior (null control)
]

# Faithfulness (patch-level insertion/deletion, Gaussian-blur baseline)
FAITH_N_STEPS = 49
N_VIZ         = 6

# Paths (Kaggle)
TRAINVAL_ROOT_STR = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
TEST_ROOT_STR     = "/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K"
PROJECT_ROOT      = "/kaggle/working/xai-vit-medical"

# ViT-B/16 checkpoint (the model being explained)
VIT_CKPT     = "/kaggle/input/models/youssefnouiouar1/<UPDATE_ME>/pytorch/default/1/vit_base_patch16_best.pth"
# ResNet-50 checkpoint (CRC-trained, builds the morphology prior)
RESNET_CKPT  = "/kaggle/input/models/youssefnouiouar1/crt-resnet50/pytorch/default/1/resnet50_best.pth"

In [ ]:
import csv
import gc
import json
import sys
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm.notebook import tqdm

assert MODEL_NAME == "vit_base", "This notebook targets ViT-B/16 (CLS readout)."

for p in (PROJECT_ROOT, f"{PROJECT_ROOT}/notebooks"):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed
from sp_lrp import DenseMorphologyPrior, SPLRPConfig, SPLRPExplainer, variant_saliency

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = list(DEFAULT_CRC_CLASSES)
NUM_CLASSES = len(CLASS_NAMES)

GRID  = IMAGE_SIZE // 16   # 14 for 224px ViT-B/16
PATCH = 16
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

SAVE_DIR = Path(f"{PROJECT_ROOT}/outputs/splrp/{MODEL_NAME}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
TRAINVAL_ROOT = Path(TRAINVAL_ROOT_STR)
TEST_ROOT     = Path(TEST_ROOT_STR)

print(f"Device  : {DEVICE}")
print(f"Classes : {CLASS_NAMES}")
print(f"Grid    : {GRID}x{GRID} patches")
print(f"Output  : {SAVE_DIR}")

In [ ]:
# ── Build ViT (explained) + ResNet-50 (morphology prior) ───────────────────

def _load_state(model: nn.Module, ckpt_path: str) -> dict:
    state = torch.load(ckpt_path, map_location="cpu")
    if isinstance(state, dict) and "model_state_dict" in state:
        model.load_state_dict(state["model_state_dict"])
        return {k: v for k, v in state.items() if k != "model_state_dict"}
    if isinstance(state, dict) and "state_dict" in state:
        model.load_state_dict({k.replace("module.", "", 1): v for k, v in state["state_dict"].items()}, strict=False)
        return {}
    model.load_state_dict({k.replace("module.", "", 1): v for k, v in state.items()}, strict=False)
    return {}


# ViT-B/16 — the model being explained
model = timm.create_model("vit_base_patch16_224", pretrained=False,
                          num_classes=NUM_CLASSES, global_pool="token", drop_path_rate=0.0)
meta = _load_state(model, VIT_CKPT)
model = model.to(DEVICE).eval()

# ResNet-50 — independent CRC morphology extractor for the semantic prior
resnet = timm.create_model("resnet50", pretrained=False, num_classes=NUM_CLASSES)
_load_state(resnet, RESNET_CKPT)
resnet = resnet.to(DEVICE).eval()

prior_extractor = DenseMorphologyPrior(resnet, grid=GRID)

print(f"ViT params    : {sum(p.numel() for p in model.parameters()):,}")
print(f"ViT ckpt epoch: {meta.get('epoch', '?')}  acc: {meta.get('val_acc', '?')}")
print(f"ResNet-50 loaded as dense morphology prior ({NUM_CLASSES} classes)")

In [ ]:
# ── Dataset & test subset ──────────────────────────────────────────────────
crc_splits = CRCSplits(trainval_root=TRAINVAL_ROOT, test_root=TEST_ROOT,
                       classes=tuple(CLASS_NAMES), val_ratio=0.25, random_state=SEED)
test_dataset = CRCHistologyDataset(split="test", splits=crc_splits,
                                   image_size=IMAGE_SIZE, return_id=True)

class_counts = defaultdict(int)
subset_indices = []
for idx, label in enumerate(test_dataset.labels):
    lbl = int(label)
    if class_counts[lbl] < N_IMAGES_PER_CLASS:
        subset_indices.append(idx); class_counts[lbl] += 1
    if all(class_counts[c] >= N_IMAGES_PER_CLASS for c in range(NUM_CLASSES)):
        break

eval_loader = DataLoader(Subset(test_dataset, subset_indices), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=torch.cuda.is_available())
print("Images per class:")
for ci, cn in enumerate(CLASS_NAMES):
    print(f"  {cn}: {class_counts[ci]}")
print(f"Total: {len(subset_indices)} | Batches: {len(eval_loader)}")

In [ ]:
# ── SP-LRP config ───────────────────────────────────────────────────────────
splrp_cfg = SPLRPConfig(tau0=TAU0, layer_dependent_tau=LAYER_DEP_TAU,
                        num_extra_tokens=1, start_layer=START_LAYER)
print(splrp_cfg)

In [ ]:
# ── Faithfulness metric (patch-level insertion/deletion, blur baseline) ────

def _blur(x: torch.Tensor, sigma_px: int = 11) -> torch.Tensor:
    k = sigma_px if sigma_px % 2 == 1 else sigma_px + 1
    coords = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g = torch.exp(-(coords ** 2) / (2 * (k / 3.0) ** 2))
    g = (g / g.sum()).view(1, 1, 1, k)
    C = x.shape[1]
    x = F.conv2d(F.pad(x, (k // 2,) * 2 + (0, 0), mode="reflect"), g.expand(C, 1, 1, k), groups=C)
    x = F.conv2d(F.pad(x, (0, 0) + (k // 2,) * 2, mode="reflect"), g.transpose(-1, -2).expand(C, 1, k, 1), groups=C)
    return x


@torch.no_grad()
def insertion_deletion_auc(model, x, sal, target, mode, n_steps=FAITH_N_STEPS, patch=PATCH, grid=GRID, fwd_bs=32):
    device = x.device
    order = torch.argsort(sal.reshape(-1), descending=True)
    n_patch = grid * grid
    per_step = max(1, n_patch // n_steps)
    steps = list(range(0, n_patch + 1, per_step))
    if steps[-1] != n_patch:
        steps.append(n_patch)
    base = _blur(x) if mode == "insertion" else torch.zeros_like(x)
    start, fill = (base, x) if mode == "insertion" else (x, base)
    masks = []
    for s in steps:
        m = torch.zeros(n_patch, device=device); m[order[:s]] = 1.0
        masks.append(F.interpolate(m.reshape(1, 1, grid, grid), scale_factor=patch, mode="nearest"))
    masks = torch.cat(masks, dim=0)
    imgs = start * (1 - masks) + fill * masks
    probs = []
    for i in range(0, imgs.shape[0], fwd_bs):
        probs.append(model(imgs[i:i + fwd_bs]).softmax(-1)[:, target])
    probs = torch.cat(probs).cpu().numpy()
    return float(np.trapz(probs, np.array(steps) / n_patch))


def overlay_pair(image_chw, saliency_hw):
    img = image_chw.detach().cpu().numpy().transpose(1, 2, 0)
    mean = IMAGENET_MEAN.view(3).numpy(); std = IMAGENET_STD.view(3).numpy()
    img = np.clip(img * std + mean, 0.0, 1.0)
    sal = saliency_hw.astype(np.float32)
    sal = (sal - sal.min()) / (sal.max() - sal.min() + 1e-8)
    sal_up = F.interpolate(torch.from_numpy(sal)[None, None], size=img.shape[:2],
                           mode="bilinear", align_corners=False)[0, 0].numpy()
    return img, sal_up


print("Evaluation helpers ready")

In [ ]:
# ── Main evaluation loop ────────────────────────────────────────────────────
# Per image: one forward+backward capture, then all variants reuse it (they differ
# only in the post-hoc modulation). Explainer hooks are removed before the
# insertion/deletion sweeps so the masked forwards do not accumulate attention.

acc = {v: {"ins": [], "del": []} for v in VARIANTS}
per_image_rows = []
viz_x, viz_y = None, None
viz_sal = {v: [] for v in VARIANTS}
n_correct = 0
n_total = 0

for bi, (images, labels, image_ids) in enumerate(tqdm(eval_loader, desc="SP-LRP eval")):
    images = images.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        logits = model(images)
        preds = logits.argmax(-1)
        priors = prior_extractor(images)          # (B, P, C)
    n_correct += int((preds.cpu() == labels).sum())
    n_total += images.shape[0]

    # ---- Phase A: compute saliencies (explainer hooks active) ----
    explainer = SPLRPExplainer(model)
    batch_sal = {v: [] for v in VARIANTS}
    for i in range(images.shape[0]):
        explainer.capture(images[i:i + 1], int(preds[i]))
        for v in VARIANTS:
            batch_sal[v].append(
                variant_saliency(explainer, v, priors[i], splrp_cfg, GRID).detach()
            )
    explainer.cleanup()                            # remove hooks BEFORE insertion/deletion
    model.zero_grad(set_to_none=True)

    # ---- Phase B: faithfulness (no hooks) ----
    for i in range(images.shape[0]):
        t = int(preds[i])
        for v in VARIANTS:
            s = batch_sal[v][i].unsqueeze(0)       # (1, grid, grid)
            ins = insertion_deletion_auc(model, images[i:i+1], s, t, "insertion")
            dele = insertion_deletion_auc(model, images[i:i+1], s, t, "deletion")
            acc[v]["ins"].append(ins); acc[v]["del"].append(dele)
            per_image_rows.append({
                "image_id": image_ids[i], "label_idx": int(labels[i]),
                "label_name": CLASS_NAMES[int(labels[i])], "pred_idx": t,
                "pred_name": CLASS_NAMES[t], "variant": v,
                "insertion_auc": ins, "deletion_auc": dele, "faithfulness": ins - dele,
            })

    if bi == 0:
        viz_x = images[:N_VIZ].detach().cpu()
        viz_y = [CLASS_NAMES[int(c)] for c in labels[:N_VIZ]]
        for v in VARIANTS:
            viz_sal[v] = [batch_sal[v][i].cpu() for i in range(min(N_VIZ, images.shape[0]))]

model_accuracy = n_correct / n_total
print(f"Model accuracy on subset: {model_accuracy:.4f} ({n_correct}/{n_total})")

In [ ]:
# ── Summary table & save ───────────────────────────────────────────────────
summary = {}
print("=" * 68)
print(f"{'Variant':<22}{'Insertion':>12}{'Deletion':>12}{'Faithfulness':>16}")
print("-" * 68)
for v in VARIANTS:
    ins = float(np.mean(acc[v]["ins"]))
    dele = float(np.mean(acc[v]["del"]))
    summary[v] = {"insertion_auc": ins, "deletion_auc": dele, "faithfulness": ins - dele}
    print(f"{v:<22}{ins:>12.4f}{dele:>12.4f}{ins - dele:>16.4f}")
print("=" * 68)
print("Higher insertion, lower deletion, higher faithfulness = better explanation.")

base = summary["chefer_ta"]["faithfulness"]
print(f"\nSP-LRP(full) vs Chefer-TA baseline: {summary['splrp_full']['faithfulness'] - base:+.4f} faithfulness")
print(f"random-prior vs baseline (should be ~0): {summary['splrp_random_prior']['faithfulness'] - base:+.4f}")

with open(SAVE_DIR / "summary.json", "w") as f:
    json.dump({"model": MODEL_NAME, "model_accuracy": model_accuracy, "n_images": n_total,
               "config": {"tau0": TAU0, "layer_dependent_tau": LAYER_DEP_TAU, "start_layer": START_LAYER},
               "results": summary}, f, indent=2)

per_image_csv = SAVE_DIR / "per_image_results.csv"
with open(per_image_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(per_image_rows[0].keys()))
    w.writeheader(); w.writerows(per_image_rows)
print(f"Saved: {SAVE_DIR / 'summary.json'}")
print(f"Saved: {per_image_csv}")

In [ ]:
# ── Qualitative overlays + faithfulness bars ───────────────────────────────
show = [v for v in ["chefer_ta", "splrp_full"] if v in VARIANTS]
n = viz_x.shape[0]
fig, axes = plt.subplots(n, len(show) + 1, figsize=(3.2 * (len(show) + 1), 3.2 * n))
axes = np.atleast_2d(axes)
mean = IMAGENET_MEAN.view(3).numpy(); std = IMAGENET_STD.view(3).numpy()
for r in range(n):
    img = np.clip(viz_x[r].permute(1, 2, 0).numpy() * std + mean, 0, 1)
    axes[r, 0].imshow(img)
    axes[r, 0].set_ylabel(viz_y[r], fontsize=9, rotation=0, labelpad=40, va="center")
    if r == 0:
        axes[r, 0].set_title("input")
    for c, v in enumerate(show):
        img_u, sal_up = overlay_pair(viz_x[r], viz_sal[v][r].numpy())
        axes[r, c + 1].imshow(img_u); axes[r, c + 1].imshow(sal_up, cmap="jet", alpha=0.5)
        if r == 0:
            axes[r, c + 1].set_title(v)
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
fig.tight_layout()
fig.savefig(SAVE_DIR / "qualitative.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
faith = [summary[v]["faithfulness"] for v in VARIANTS]
colors = ["tab:gray" if v in ("chefer_ta", "splrp_random_prior") else "tab:blue" for v in VARIANTS]
ax.bar(range(len(VARIANTS)), faith, color=colors)
ax.set_xticks(range(len(VARIANTS))); ax.set_xticklabels(VARIANTS, rotation=30, ha="right")
ax.set_ylabel("Faithfulness (Insertion − Deletion)")
ax.axhline(summary["chefer_ta"]["faithfulness"], color="k", ls="--", lw=0.8, label="Chefer-TA baseline")
ax.set_title(f"{MODEL_NAME} — SP-LRP ablation (higher = better)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(SAVE_DIR / "faithfulness_bars.png", dpi=150)
plt.show()
print(f"Saved: {SAVE_DIR / 'qualitative.png'}")
print(f"Saved: {SAVE_DIR / 'faithfulness_bars.png'}")

In [ ]:
# ── Upload results to Google Drive ─────────────────────────────────────────
RESULTS_DRIVE_FOLDER = DRIVE_FOLDER_ID
files_to_upload = [SAVE_DIR / "summary.json", SAVE_DIR / "per_image_results.csv",
                   SAVE_DIR / "qualitative.png", SAVE_DIR / "faithfulness_bars.png"]
for fpath in files_to_upload:
    fpath = Path(fpath)
    if not fpath.exists():
        print(f"  Not found (skipped): {fpath.name}"); continue
    drive_file = drive.CreateFile({"title": f"splrp_{MODEL_NAME}_{fpath.name}",
                                   "parents": [{"id": RESULTS_DRIVE_FOLDER}]})
    drive_file.SetContentFile(str(fpath)); drive_file.Upload()
    print(f"  Uploaded: splrp_{MODEL_NAME}_{fpath.name}")
print("Done.")

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleanup done.")